In [15]:
%matplotlib inline

import os
import sys
import math
import time

from sys import platform
from pathlib import Path

sys.path.append('../../')

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from importlib.metadata import version 

import tiktoken

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

%load_ext autoreload
%autoreload 2

from llm_from_scratch.CH4.gpt import GPTModel, generate_text_simple
from llm_from_scratch.CH5.utils import load_weights_into_gpt, text_to_token_ids, token_ids_to_text
from llm_from_scratch.CH5.gpt_download import download_and_load_gpt2

from spam_dataset import SpamDataset, download_and_unzip_spam_data, create_balanced_dataset, random_split
from utils import calc_accuracy_loader, calc_loss_loader, calc_loss_batch

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from importlib.metadata import version

pkgs=["matplotlib",   # plotting library
      "numpy",        # Pytorch and TensorFlow dependency
      "tiktoken",     # tokenizer
      "torch",        # deep learning library
      "tensorflow",   # for openAI's pretrained weights
      "pandas"
     ]
for p in pkgs: print(f"{p} version: {version(p)}")

matplotlib version: 3.10.9
numpy version: 2.2.6
tiktoken version: 0.12.0
torch version: 2.12.0
tensorflow version: 2.21.0
pandas version: 2.3.3


In [3]:
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
main_dirpath="/Users/reaungamornrat.sureerat/data/sms_spam_collection"
zip_path=f"{main_dirpath}/sms_spam_collection.zip"
extracted_path=main_dirpath
data_file_path=Path(extracted_path)/"SMSSpamCollection.tsv"
try: download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (requests.exceptions.RequestException, TimeoutError) as e:
    print(f"Primary YRL failed: {e}. Trying backup URL....")
    url="https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

/Users/reaungamornrat.sureerat/data/sms_spam_collection/SMSSpamCollection.tsv already exists. Skipping download and extraction


In [4]:
df=pd.read_csv(data_file_path, sep='\t', header=None, names=["Label", "Text"])
print(f'{df["Label"].value_counts()}')
df

Label
ham     4825
spam     747
Name: count, dtype: int64


,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [5]:
balanced_df=create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())
# change the string class labels "ham" and "spam" into integer class labels 0 and 1
balanced_df["Label"]=balanced_df["Label"].map({"ham":0, "spam":1})
balanced_df

Label
ham     747
spam    747
Name: count, dtype: int64


,Label,Text
4307,0,Awww dat is sweet! We can think of something t...
4138,0,Just got to &lt;#&gt;
4831,0,"The word ""Checkmate"" in chess comes from the P..."
4461,0,This is wishing you a great day. Moji told me ...
5440,0,Thank you. do you generally date the brothas?
...,...,...
5537,1,Want explicit SEX in 30 secs? Ring 02073162414...
5540,1,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547,1,Had your contract mobile 11 Mnths? Latest Moto...
5566,1,REMINDER FROM O2: To get 2.50 pounds free call...


In [6]:
# split data into 70% training, 10% validation, and 20% testing
train_df, validation_df, test_df=random_split(balanced_df, 0.7, 0.1)
train_df.to_csv(f"{main_dirpath}/train.csv", index=None)
validation_df.to_csv(f"{main_dirpath}/validation.csv", index=None)
test_df.to_csv(f"{main_dirpath}/test.csv", index=None)

tokenizer=tiktoken.get_encoding("gpt2")
pad_token_id=tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]
print(f'{pad_token_id=}')
training_dataset=SpamDataset(csv_file=f"{main_dirpath}/train.csv", max_length=None, tokenizer=tokenizer, pad_token_id=pad_token_id)
print(f"{training_dataset.max_length=}")
# Note that this model can handles sequences up to 1024 tokens so we do not need to truncate the validation and test text to 
# `training_dataset.max_length` as long as the sequence lengths of the validation and test sets are < 1024 tokens.
val_dataset=SpamDataset(csv_file=f"{main_dirpath}/validation.csv", max_length=training_dataset.max_length, tokenizer=tokenizer,
                        pad_token_id=pad_token_id)
test_dataset=SpamDataset(csv_file=f"{main_dirpath}/test.csv", max_length=training_dataset.max_length, tokenizer=tokenizer,
                         pad_token_id=pad_token_id)

num_workers=0
batch_size=8
torch.manual_seed(123)
train_loader=DataLoader(dataset=training_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
val_loader=DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=False)
test_loader=DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=False)
print("Train loader:")
for input_batch, target_batch in train_loader: pass
print("Input batch dimensions: ", input_batch.shape)
print("Label batch dimensions: ", target_batch.shape)
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} training batches")
print(f"{len(test_loader)} training batches")

pad_token_id=50256
training_dataset.max_length=120
Train loader:
Input batch dimensions:  torch.Size([8, 120])
Label batch dimensions:  torch.Size([8])
130 training batches
19 training batches
38 training batches


In [7]:
CHOOSE_MODEL="gpt2-small (124M)"
INPUT_PROMPT="Every effort moves"
BASE_CONFIG={"vocab_size":50257,
             "context_length":1024,
             "drop_rate":0.,
             "qkv_bias":True}
model_configs={
    "gpt2-small (124M)":{"emb_dim":768, "n_layers":12, "n_heads":12},
    "gpt2-medium (355M)":{"emb_dim":1024, "n_layers":24, "n_heads":16},
    "gpt2-large (774M)":{"emb_dim":1280, "n_layers":36, "n_heads":20},
    "gpt2-xl (1558M)":{"emb_dim":1600, "n_layers":48, "n_heads":25},
}
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
assert training_dataset.max_length<=BASE_CONFIG["context_length"], (
    f"Dataset length {training_dataset.max_length} exceeds model's context"
    f"length {BASE_CONFIG['context_length']}. Reinitialize data sets with max_length={BASE_CONFIG['context_length']}"
)

model_size=CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
print(f"{model_size=}")
model_dir='/Users/reaungamornrat.sureerat/data/GPT/GPT2'
settings, params=download_and_load_gpt2(model_size=model_size,models_dir=model_dir)
print(f'{os.listdir(model_dir)=}')

model=GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

model_size='124M'
File already exists and is up-to-date: /Users/reaungamornrat.sureerat/data/GPT/GPT2/124M/checkpoint
File already exists and is up-to-date: /Users/reaungamornrat.sureerat/data/GPT/GPT2/124M/encoder.json
File already exists and is up-to-date: /Users/reaungamornrat.sureerat/data/GPT/GPT2/124M/hparams.json
File already exists and is up-to-date: /Users/reaungamornrat.sureerat/data/GPT/GPT2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: /Users/reaungamornrat.sureerat/data/GPT/GPT2/124M/model.ckpt.index
File already exists and is up-to-date: /Users/reaungamornrat.sureerat/data/GPT/GPT2/124M/model.ckpt.meta
File already exists and is up-to-date: /Users/reaungamornrat.sureerat/data/GPT/GPT2/124M/vocab.bpe
os.listdir(model_dir)=['124M']


In [8]:
# Test whether the model was loaded correctly
text_1="Every effort moves you"
token_ids=generate_text_simple(model=model,idx=text_to_token_ids(text_1, tokenizer), max_new_tokens=15, 
                               context_size=BASE_CONFIG['context_length'])
print(token_ids_to_text(token_ids, tokenizer))

Every effort moves you forward.

The first step is to understand the importance of your work


In [9]:
# Before we finetune the model as a classifier, let's see whether the model can already 
# classify spam messages by prompting it with instruction
text_2=("Is the following text 'spam'? Answer with 'yes' or 'no': "
       " 'You are a winner you have been specially selected to receive $1000 cash or a $2000 reward.'")
tokens_id=generate_text_simple(model=model, idx=text_to_token_ids(text_2, tokenizer), max_new_tokens=23,
                               context_size=BASE_CONFIG['context_length'])
print(token_ids_to_text(tokens_id, tokenizer))

Is the following text 'spam'? Answer with 'yes' or 'no':  'You are a winner you have been specially selected to receive $1000 cash or a $2000 reward.'

'You are a winner you have been specially selected to receive $1000 cash or a $2000 reward.'


In [10]:
# look at model archiutecture first
print(model)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

In [11]:
# we freeze the model
for param in model.parameters(): param.requires_grad=False
torch.manual_seed(123)
num_classes=2
model.out_head=torch.nn.Linear(in_features=BASE_CONFIG['emb_dim'], out_features=num_classes)
for param in model.trf_blocks[-1].parameters(): param.requires_grad=True
for param in model.final_norm.parameters(): param.requires_grad=True

In [12]:
inputs=tokenizer.encode("Do you have time")
inputs=torch.tensor(inputs).unsqueeze(0)
print("Inputs: ", inputs)
print("Inputs dimensions: ", inputs.shape)

with torch.no_grad(): outputs=model(inputs)
print(f"Outputs: {outputs.shape=}\n{outputs=}")
print(f"Last output token: {outputs[:,-1]}")

Inputs:  tensor([[5211,  345,  423,  640]])
Inputs dimensions:  torch.Size([1, 4])
Outputs: outputs.shape=torch.Size([1, 4, 2])
outputs=tensor([[[-1.5854,  0.9904],
         [-3.7235,  7.4548],
         [-2.2661,  6.6049],
         [-3.5983,  3.9902]]])
Last output token: tensor([[-3.5983,  3.9902]])


In [13]:
probs=torch.softmax(outputs[:,-1], dim=-1)
label=torch.argmax(probs)
print(f"{probs=}, {label=}")
label=torch.argmax(outputs[:,-1])
print(label)

probs=tensor([[5.0598e-04, 9.9949e-01]]), label=tensor(1)
tensor(1)


In [14]:
device=torch.device('cuda') if torch.cuda.is_available() else torch.device('mps') if torch.mps.is_available() else torch.device('cpu')
model.to(device)
torch.manual_seed(123)
train_accuracy=calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy=calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy=calc_accuracy_loader(test_loader, model, device, num_batches=10)
print(f"Train accuracy: {train_accuracy*100:.2f}%")
print(f"Val accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Train accuracy: 46.25%
Val accuracy: 45.00%
Test accuracy: 48.75%


In [17]:
with torch.no_grad():
    train_loss=calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss=calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss=calc_loss_loader(test_loader, model, device, num_batches=5)
print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

Training loss: 3.211
Validation loss: 2.583
Test loss: 2.322


In [39]:
torch.argmax(x, dim=-1)

tensor([2, 2, 0])